# 04 — Integer Capacity Expansion Optimization

## Portfolio objective

Determine the lowest-cost timing and quantity of standardized server additions while maintaining sufficient planning capacity.

### Model structure

**Decision variable**

`new_servers[t]` = integer number of new servers added in period t.

**Objective**

Minimize:

`CAPEX + recurring OPEX + shortage penalty`

**Constraint**

`baseline capacity + installed expansion capacity + shortage >= required planning capacity`

Headroom is **not** multiplied again here because Notebook 03 has already applied the planning assumptions.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from cpoe.config import PROCESSED, FIGURES
from cpoe.optimization.model import optimize_capacity

capacity = pd.read_csv(
    PROCESSED / "capacity_requirements.csv",
    index_col=0,
    parse_dates=True,
)

solution, summary = optimize_capacity(capacity)

solution.to_csv(PROCESSED / "optimization_solution.csv")
summary.to_csv(PROCESSED / "optimization_summary.csv")

display(summary)


In [ ]:
display(solution.head(10))

print("Total new servers:", solution["new_servers"].sum())
print("Total CAPEX:", solution["capex"].sum())
print("Total OPEX:", solution["period_opex"].sum())
print("Peak shortage:", solution["shortage"].max())


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(solution.index, solution["required_capacity"], label="Required planning capacity")
plt.plot(solution.index, solution["available_capacity"], label="Optimized available capacity")
plt.title("Optimized Capacity Expansion Plan")
plt.xlabel("Trace-relative time")
plt.ylabel("GPU-equivalent capacity")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES / "optimized_capacity_plan.png", dpi=150)
plt.show()


## Decision question

> What is the lowest-cost timing of standardized server additions while maintaining sufficient planning capacity?

Notebook 05 stress-tests whether that optimized plan remains resilient when demand or available capacity changes.
